In [1]:
import pandas as pd
import matplotlib.pyplot as plt

PROCESSED_PATH = "C:/Users/kalam/F1 Project Outcome Prediction/data/processed/"
f1 = pd.read_csv(PROCESSED_PATH + "f1_merged.csv")

In [2]:
f1["date"] = pd.to_datetime(f1["date"], errors="coerce")
f1["positionOrder"] = pd.to_numeric(f1["positionOrder"], errors="coerce")
f1["grid"] = pd.to_numeric(f1["grid"], errors="coerce")
f1["points"] = pd.to_numeric(f1["points"], errors="coerce")
f1["quali_position"] = pd.to_numeric(f1["quali_position"], errors="coerce")

In [3]:
f1 = f1.sort_values(["driverId", "date"]).reset_index(drop=True)

In [4]:
f1["is_win"] = (f1["positionOrder"] == 1).astype(int)
f1["is_podium"] = (f1["positionOrder"] <= 3).astype(int)

finished_keywords = ["Finished", "+1 Lap", "+2 Laps", "+3 Laps", "+4 Laps", "+5 Laps", "+6 Laps", "+7 Laps"]
f1["is_dnf"] = (~f1["race_status"].isin(finished_keywords)).astype(int)

In [5]:
f1["driver_experience"] = f1.groupby("driverId").cumcount()

In [6]:
f1["prev_wins"] = f1.groupby("driverId")["is_win"].cumsum() - f1["is_win"]
f1["prev_podiums"] = f1.groupby("driverId")["is_podium"].cumsum() - f1["is_podium"]

In [7]:
f1["prev_avg_finish"] = (
    f1.groupby("driverId")["positionOrder"]
      .transform(lambda x: x.shift().expanding().mean())
)

In [8]:
f1["recent_avg_finish_5"] = (
    f1.groupby("driverId")["positionOrder"]
      .transform(lambda x: x.shift().rolling(5, min_periods=1).mean())
)

In [9]:
f1 = f1.sort_values(["constructorId", "date"]).reset_index(drop=True)
f1["constructor_prev_avg_points"] = (
    f1.groupby("constructorId")["points"]
      .transform(lambda x: x.shift().expanding().mean())
)

In [10]:
fill_cols = [
    "driver_experience", "prev_wins", "prev_podiums",
    "prev_avg_finish", "recent_avg_finish_5",
    "constructor_prev_avg_points", "pit_stop_count",
    "mean_pit_stop_duration_ms", "avg_lap_time_ms", "quali_position"
]

for col in fill_cols:
    f1[col] = f1[col].fillna(0)

In [12]:
feature_cols = [
    "year", "round", "grid", "quali_position",
    "driver_experience", "prev_wins", "prev_podiums",
    "prev_avg_finish", "recent_avg_finish_5",
    "constructor_prev_avg_points", "pit_stop_count",
    "mean_pit_stop_duration_ms", "avg_lap_time_ms"
]

In [12]:
model_df = f1[[
    "raceId", "driverId", "driver_name", "constructor_name",
    "positionOrder", "is_win", "is_podium", "is_dnf"
] + feature_cols].copy()

model_df.to_csv(PROCESSED_PATH + "f1_features.csv", index=False)
print("Saved: f1_features.csv")
model_df.head()

Saved: f1_features.csv


,raceId,driverId,driver_name,constructor_name,positionOrder,is_win,is_podium,is_dnf,year,round,...,quali_position,driver_experience,prev_wins,prev_podiums,prev_avg_finish,recent_avg_finish_5,constructor_prev_avg_points,pit_stop_count,mean_pit_stop_duration_ms,avg_lap_time_ms
0,674,360,Bruce McLaren,McLaren,13,0,0,0,1968,8,...,0.0,84,4,21,8.964286,9.6,0.000000,0.0,0.0,0.0
1,677,304,Denny Hulme,McLaren,8,0,0,1,1968,11,...,0.0,36,4,15,7.027778,3.6,0.000000,0.0,0.0,0.0
2,632,304,Denny Hulme,McLaren,6,0,0,0,1971,1,...,0.0,60,5,21,7.233333,9.0,0.000000,0.0,0.0,0.0
3,632,320,Peter Gethin,McLaren,23,0,0,1,1971,1,...,0.0,7,0,0,13.000000,10.4,0.333333,0.0,0.0,0.0
4,632,347,Jo Bonnier,McLaren,24,0,0,1,1971,1,...,0.0,104,1,1,11.615385,14.6,0.250000,0.0,0.0,0.0
